# ĆW 1

- wczytaj zbior tytanic, wyswietl wymiary
- utworz kopie, wprowadz zanieczyszczenia, 4 różne typy błędów
- przeprowadź audyt: braki, duplikaty, zakresy wartości, unikalne wartości zmiennych kategorycznych
- zidentyfikuj powtarzające się informacje w kolumnach: survived/alive, pclass/class, embarked/embark_town
- wyczyść zbiór 

In [40]:
import numpy as np
import pandas as pd
import seaborn as sns

df_titanic = sns.load_dataset("titanic")

print(df_titanic.info())
print(df_titanic.shape)
print(df_titanic.describe())

df = df_titanic.copy()

df.loc[3, "sex"] = "N"
df.loc[10, "age"] = -1
df.loc[15, "who"] = "brak"

df = pd.concat(
  [df, df.iloc[[0]]], ignore_index=True
)

print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.duplicated().sum())

print(df.isna().sum())

df["age"] = df["age"].fillna(df["age"].mean())

df['deck'] = df['deck'].replace('', np.nan)

df = df.dropna(subset=['embark_town', 'deck'])

print(df.isna().sum())

print(df.describe())

df['age'] = np.where(
  df['age'] > 0, df['age'], np.nan
)

df.dropna(subset=["age"])

print(df.describe())

print(df["sex"].value_counts())
print(df["embarked"].value_counts())
print(df["who"].value_counts())
print(df["embark_town"].value_counts())
print(df["deck"].value_counts())

df['sex'] = np.where(
  df['sex'] != 'N', df['sex'], np.nan
)

df.dropna(subset=["sex"])
print(df["sex"].value_counts())



<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB
None
(891, 15)
         survived      pclass         age       sibsp       parch        fare
c

In [38]:
print(df["survived"].value_counts())
print(df["alive"].value_counts())
print(df["embarked"].value_counts())
print(df["embark_town"].value_counts())
print(df["pclass"].value_counts())
print(df["class"].value_counts())

df = df.drop(columns=["alive", "embark_town", "class"])

df.info()

survived
1    133
0     67
Name: count, dtype: int64
alive
yes    133
no      67
Name: count, dtype: int64
embarked
S    128
C     68
Q      4
Name: count, dtype: int64
embark_town
Southampton    128
Cherbourg       68
Queenstown       4
Name: count, dtype: int64
pclass
1    172
2     16
3     12
Name: count, dtype: int64
class
First     172
Second     16
Third      12
Name: count, dtype: int64
<class 'pandas.DataFrame'>
Index: 200 entries, 1 to 889
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   survived    200 non-null    int64   
 1   pclass      200 non-null    int64   
 2   sex         199 non-null    str     
 3   age         199 non-null    float64 
 4   sibsp       200 non-null    int64   
 5   parch       200 non-null    int64   
 6   fare        200 non-null    float64 
 7   embarked    200 non-null    str     
 8   who         200 non-null    str     
 9   adult_male  200 non-null    bool    
 10  deck

### ĆW 2

- Zdefiniuj schemat DataFramePandera aby wyczyścić zanieczyszczony zbiór 
- Sprawdź jakie błędy wyrzuca na brudnym
- Zwaliduj oczyszczony zbiór

In [45]:
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

schemat = DataFrameSchema({
"sex": Column(str, Check.isin(["male", "female"])),
"deck": Column(str, Check.isin(["A", "B", "C", "D", "E", "F", "G"])),
"age": Column(float, Check.in_range(0, 100), nullable=True)
})


try:
  schemat.validate(df, lazy=True)
  print("Walidacja zakończona powodzeniem")
except pa.errors.SchemaErrors as err:
  print(err.failure_cases)

validated = schemat.validate(df_titanic) 
print("Walidacja zakończona powodzeniem, wymiary:", validated.shape) 

  schema_context column                    check check_number failure_case  \
0         Column    sex             not_nullable         None          NaN   
1         Column   deck  dtype('string[python]')         None     category   

  index  
0     3  
1  None  


SchemaError: non-nullable series 'deck' contains null values:
0      NaN
2      NaN
4      NaN
5      NaN
7      NaN
      ... 
884    NaN
885    NaN
886    NaN
888    NaN
890    NaN
Name: deck, Length: 688, dtype: category
Categories (7, str): ['A', 'B', 'C', 'D', 'E', 'F', 'G']

### ĆW 3

- Zakoduj zmienne z OneHotEncoder
- Zakoduj zmienną class z OrdinalEncoder, porównaj z pclass
- Przeskaluj age i fare z StandardScaler i MinMaxScaler, porównaj. 
  - Sprawdź wartości odstające w fare, przetestuj RobustScaler

In [55]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder

oh_encoder = OneHotEncoder()
std_scaler = StandardScaler()
min_scaler = MinMaxScaler()
or_encoder = OrdinalEncoder()

X = df_titanic[["embarked"]]
X_fit = oh_encoder.fit_transform(X)
print(X, X_fit.toarray())
print(oh_encoder.get_feature_names_out())

X = df_titanic[["class"]]
X_fit = or_encoder.fit_transform(X)
print(X, X_fit)

X = df_titanic[["fare", "age"]].dropna()
X_fit = std_scaler.fit_transform(X)
print(X, X_fit)

X = df_titanic[["fare", "age"]].dropna()
X_fit = min_scaler.fit_transform(X)
print(X, X_fit)


    embarked
0          S
1          C
2          S
3          S
4          S
..       ...
886        S
887        S
888        S
889        C
890        Q

[891 rows x 1 columns] [[0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 ...
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]
['embarked_C' 'embarked_Q' 'embarked_S' 'embarked_nan']
      class
0     Third
1     First
2     Third
3     First
4     Third
..      ...
886  Second
887   First
888   Third
889   First
890   Third

[891 rows x 1 columns] [[2.]
 [0.]
 [2.]
 [0.]
 [2.]
 [2.]
 [0.]
 [2.]
 [2.]
 [1.]
 [2.]
 [0.]
 [2.]
 [2.]
 [2.]
 [1.]
 [2.]
 [1.]
 [2.]
 [2.]
 [1.]
 [1.]
 [2.]
 [0.]
 [2.]
 [2.]
 [2.]
 [0.]
 [2.]
 [2.]
 [0.]
 [0.]
 [2.]
 [1.]
 [0.]
 [0.]
 [2.]
 [2.]
 [2.]
 [2.]
 [2.]
 [1.]
 [2.]
 [1.]
 [2.]
 [2.]
 [2.]
 [2.]
 [2.]
 [2.]
 [2.]
 [2.]
 [0.]
 [1.]
 [0.]
 [0.]
 [1.]
 [2.]
 [1.]
 [2.]
 [2.]
 [0.]
 [0.]
 [2.]
 [0.]
 [2.]
 [1.]
 [2.]
 [2.]
 [2.]
 [1.]
 [2.]
 [1.]
 [2.]
 [2.]
 [2.]
 [2.]
 [2.]
 [1.]
 [2.]
 [2.]
 [2.]
 [2.]
 [

### ĆW 4

- Przygotuj macierz z: age, fare, sibsp, parch i zakodowaną klasę podróży (porządkowo)
- Oblicz macierz korelacji, wyświetl
- Wykonaj standaryzacje, wykonaj PCA, liczba składowych do zachowania 80% wariancji
- Przeanalizuj PC1 i PC2, wskaż cechy które miały największy wpływ

In [64]:
from sklearn.decomposition import PCA

df_temp = df[["age", "fare", "sibsp", "parch", "pclass"]].dropna()

print(df_temp.corr())

pca = PCA()

std_scaler = StandardScaler()

X = std_scaler.fit_transform(df_temp)

X = pd.DataFrame(pca.fit_transform(X))

print(X)
print(pca.explained_variance_ratio_.round(3))

values = pd.DataFrame(pca.components_[:2])
print(values.round(2))

             age      fare     sibsp     parch    pclass
age     1.000000 -0.085230 -0.133832 -0.243905 -0.263268
fare   -0.085230  1.000000  0.295001  0.388252 -0.309390
sibsp  -0.133832  0.295001  1.000000  0.256738 -0.108230
parch  -0.243905  0.388252  0.256738  1.000000  0.042526
pclass -0.263268 -0.309390 -0.108230  0.042526  1.000000
            0         1         2         3         4
0    0.080938 -0.516274  1.000096 -0.171060  0.067546
1   -0.007564 -0.331907  1.076768 -0.285076 -0.129723
2   -1.143772 -1.081061 -0.271520  0.379995  0.060133
3   -1.414338 -1.164815 -0.165536  0.619704 -0.133785
4   -1.454533  1.259824 -0.008446  0.314729  0.511880
..        ...       ...       ...       ...       ...
194  0.521767 -0.521421  0.514099  0.973259 -0.672517
195 -1.132723 -0.053494 -0.072704 -0.560415 -0.579225
196 -0.184539 -0.958371 -0.967057  1.094689 -0.287757
197 -0.684372  0.447619 -0.176460 -1.286871 -0.486883
198 -0.810536  0.156183 -0.177101 -0.945761 -0.417797

[199 rows

### ĆW 5

- Augumentacja metodą jitter dla age i fare na zbiorze treningowym, szum 2% odchylenia. Zachowaj etykiety survived, podaj rozmiary zbioru
- Wynik na wykresie punktowym age-fare, oryginalne i augumentowane innymi znacznikami.
- Powtórz augumetnacje z szumem 50%

### ĆW 6

- Podziel zbiór na trening i test, względem survived
- ColumnTransformer, przygotuj potok
  - imputacja medianą i skalowanie dla liczbowych
  - imputacja dominantą i one-hot dla kategorycznych 
  - na końcu PCA
- Potok na zbiorze treningowym i przekształć, podaj wymiary cech
- Zapisz potok do pliku za pomocą joblib i go wczytaj